# Lab 2: Minimal Pair practice with Implicit Causality
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer the questions in `Lab2.md`. For answers where you are looking at data files generated by the `evaluate` or `analyze` modes, you should load the data into the ipynb notebook and proceed from there. 

## Part 1

#### Question 1

We can evaluate `distilgpt2`'s bias by testing its ability to predict the next word in a carefully constructed sentence. For example, we can create a minimal pair of sentences:

- John frightened Mary because **he** was so terrifying.
- John frightened Mary because **she** was so terrifying.

To evaluate the model's bias for *frightened*, we would provide the prefix "John frightened Mary because" and check the probability the model assigns to the next word being "he" versus the probability of it being "she". If the model has the same subject-implicit causality bias as humans, it should assign a higher probability to "he" than to "she".


#### Question 2

The target words are the pronouns "he" and "she" that appear after "because" in each sentence. We used these two pronouns in the testing because unlike names, which may vary in the use frequency, the pronouns are more fair in comparison.


#### Question 3

- Expected sentence: John frightened Mary because he was so terrifying.
- Unexpected sentence: John frightened Mary because she was so terrifying.

This is because *frightened* biases toward subject causality.

## Part 2


#### Results from experiment setup in Part 1

In [14]:
import pandas as pd

df = pd.read_csv('results/ic_bypair.tsv', sep='\t')

# Map pairid to verb conditions
verb_mapping = {
    1: 'frightened (Subject IC)',
    2: 'feared (Object IC)', 
    3: 'bored (Subject IC)',
    4: 'believed (Object IC)'
}

df['verb_condition'] = df['pairid'].map(verb_mapping)

print("Question 1: Does distilgpt2 learn the subject IC verb bias?")

subject_ic_results = df[df['pairid'].isin([1, 3])]
for _, row in subject_ic_results.iterrows():
    verb = row['verb_condition'].split()[0]
    expected_prob = row['expected']
    unexpected_prob = row['unexpected']
    correct = row['acc']
    
    print(f"  {verb}:")
    print(f"    Expected (subject reference): {expected_prob:.4f}")
    print(f"    Unexpected (object reference): {unexpected_prob:.4f}")
    print(f"    Model shows expected bias: {correct}")

print()    
print("Question 2: Does distilgpt2 learn the object IC verb bias?")

object_ic_results = df[df['pairid'].isin([2, 4])]
for _, row in object_ic_results.iterrows():
    verb = row['verb_condition'].split()[0]
    expected_prob = row['expected']
    unexpected_prob = row['unexpected']
    correct = row['acc']
    
    print(f"  {verb}:")
    print(f"    Expected (object reference): {expected_prob:.4f}")
    print(f"    Unexpected (subject reference): {unexpected_prob:.4f}")
    print(f"    Model shows expected bias: {correct}")

Question 1: Does distilgpt2 learn the subject IC verb bias?
  frightened:
    Expected (subject reference): 0.0309
    Unexpected (object reference): 0.4394
    Model shows expected bias: False
  bored:
    Expected (subject reference): 0.0389
    Unexpected (object reference): 0.4042
    Model shows expected bias: False

Question 2: Does distilgpt2 learn the object IC verb bias?
  feared:
    Expected (object reference): 0.3349
    Unexpected (subject reference): 0.0310
    Model shows expected bias: True
  believed:
    Expected (object reference): 0.3742
    Unexpected (subject reference): 0.0580
    Model shows expected bias: True


#### Results from experiment setup with same target word
- Expected: Sally frightened Mary because **she** was so terrifying.
- Unexpected: John frightened Mary because **she** was so terrifying.

In [13]:
df = pd.read_csv('results/ic_same_bypair.tsv', sep='\t')

# Map pairid to verb conditions
verb_mapping = {
    1: 'frightened (Subject IC)',
    2: 'feared (Object IC)', 
    3: 'bored (Subject IC)',
    4: 'believed (Object IC)'
}

df['verb_condition'] = df['pairid'].map(verb_mapping)

print("Question 1: Does distilgpt2 learn the subject IC verb bias?")

subject_ic_results = df[df['pairid'].isin([1, 3])]
for _, row in subject_ic_results.iterrows():
    verb = row['verb_condition'].split()[0]
    expected_prob = row['expected']
    unexpected_prob = row['unexpected']
    correct = row['acc']
    
    print(f"  {verb}:")
    print(f"    Expected (subject reference): {expected_prob:.4f}")
    print(f"    Unexpected (object reference): {unexpected_prob:.4f}")
    print(f"    Model shows expected bias: {correct}")

print()    
print("Question 2: Does distilgpt2 learn the object IC verb bias?")

object_ic_results = df[df['pairid'].isin([2, 4])]
for _, row in object_ic_results.iterrows():
    verb = row['verb_condition'].split()[0]
    expected_prob = row['expected']
    unexpected_prob = row['unexpected']
    correct = row['acc']
    
    print(f"  {verb}:")
    print(f"    Expected (object reference): {expected_prob:.4f}")
    print(f"    Unexpected (subject reference): {unexpected_prob:.4f}")
    print(f"    Model shows expected bias: {correct}")

Question 1: Does distilgpt2 learn the subject IC verb bias?
  frightened:
    Expected (subject reference): 0.4601
    Unexpected (object reference): 0.4394
    Model shows expected bias: True
  bored:
    Expected (subject reference): 0.3988
    Unexpected (object reference): 0.4042
    Model shows expected bias: False

Question 2: Does distilgpt2 learn the object IC verb bias?
  feared:
    Expected (object reference): 0.3457
    Unexpected (subject reference): 0.0310
    Model shows expected bias: True
  believed:
    Expected (object reference): 0.4121
    Unexpected (subject reference): 0.0580
    Model shows expected bias: True


## Part 3

#### Question 1

We give the following example to test gender bias:

- Men frightened Mary because they were so terrifying.

- Women frightened Mary because they were so terrifying.

We can test this by varying only the gender of the subject NP while keeping everything else constant. By comparing the probability that the model assigns to "they" in each context, we can detect whether the model has different baseline preferences for attributing causality based on the gender of the subject, independent of pronoun choice. 

Or we could use the "difference between differences" method:

- Pair1 - Expected: John frightened Mary because he was so terrifying.

- Pair1 - Unexpected: Sally frightened Mary because he was so terrifying.

- Pair2 - Expected: Sally frightened Mary because she was so terrifying.

- Pair2 - Unexpected: John frightened Mary because she was so terrifying.

We can calculate the differences between Pair1 "he" and Pair2 "she" and make a comparison between these differences.

#### Question 2

- John frightened Mary because he was so terrifying.
- The man frightened Mary because he was so terrifying.

This varies only the specificity of the NP (proper noun versus generic noun phrase). By comparing the probability of "he" in each context, we can determine whether names create stronger referential links than generic NPs and whether the model processes proper nouns differently than common nouns in causality attribution.

#### Question 3

To study whether the model's IC verb bias aligns with human IC verb bias, we would need to significantly expand the current approach beyond the binary classification of just four verbs. As we currently have, we should have human-labelled data as the gold standard. We need to modify the approach by testing a much larger set of verbs spanning the entire IC spectrum. Furthermore, instead of measuring binary accuracy, we could compute continuous bias strength scores by calculating the absolute difference between subject and object reference probabilities (assuming in the gold data, there could be some differences between strong IC and neutral verbs). In the current setting, we could calculate the accuracy of the model prediction to see if it reaches a certain threshold. Given continuous bias strength scores, we could map a correlation to human data to see if the results align.